<a href="https://colab.research.google.com/github/yashb98/90Days_Machine_learinng/blob/main/Mistral_7B_Manual_factual_check_and_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install -q streamlit # -q for "quiet"
!pip install -q langchain langchain-openai llama-index openai faiss-cpu sentence-transformers pandas python-dotenv
!pip install -q openai-whisper

# Install ffmpeg for audio processing
!apt-get install -y -qq ffmpeg
!pip install datasets

In [ ]:

import os
from datasets import load_dataset

print("Creating directory structure...")
os.makedirs("data/ehr", exist_ok=True)
os.makedirs("data/golden_path", exist_ok=True)
!ls -R data

print("\nDownloading MTS-Dialog dataset from Hugging Face...")
# This dataset has 'train', 'validation', 'test' splits. We'll use 'train'.
try:
    mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note", split='train')
    print("\nMTS-Dialog dataset loaded successfully.")
    print(f"Total samples: {len(mts_dataset)}")

    # Let's inspect the first sample
    print("\n--- Sample 1 ---")
    print(f"[DIALOGUE]:\n{mts_dataset[0]['dialogue']}")
    print(f"\n[NOTE]:\n{mts_dataset[0]['note']}")
    print("------------------")

except Exception as e:
    print(f"Error loading dataset: {e}")


In [ ]:

import os
from datasets import load_dataset

print("Creating directory structure...")
os.makedirs("data/ehr", exist_ok=True)
os.makedirs("data/golden_path", exist_ok=True)
!ls -R data

print("\nDownloading MTS-Dialog dataset from Hugging Face...")
# This dataset has 'train', 'validation', 'test' splits. We'll use 'train'.
try:
    mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note", split='train')
    print("\nMTS-Dialog dataset loaded successfully.")
    print(f"Total samples: {len(mts_dataset)}")

    # Let's inspect the first sample
    print("\n--- Sample 1 ---")
    print(f"[DIALOGUE]:\n{mts_dataset[0]['dialogue']}")
    print(f"\n[NOTE]:\n{mts_dataset[0]['note']}")
    print("------------------")

except Exception as e:
    print(f"Error loading dataset: {e}")


In [ ]:

from google.colab import drive
import os

print("Mounting Google Drive...")
drive.mount('/content/drive')

# --- VERIFY YOUR PATH ---
# This command lists the files in your 'csv' folder.
# If this command fails, your folder isn't at 'My Drive/csv'.
# Adjust the path as needed.
print("\nVerifying access to your Synthea files...")
!ls -lh /content/drive/MyDrive/csv

In [ ]:

import pandas as pd
from pathlib import Path
import os

# Path to your Synthea CSVs in Google Drive
CSV_DIR = Path("/content/drive/MyDrive/csv")

# List of all 18 CSV files from your screenshot
csv_files_list = [
    "allergies.csv",
    "careplans.csv",
    "claims_transactions.csv",
    "claims.csv",
    "conditions.csv",
    "devices.csv",
    "encounters.csv",
    "imaging_studies.csv",
    "immunizations.csv",
    "medications.csv",
    "observations.csv",
    "organizations.csv",
    "patients.csv",
    "payer_transitions.csv",
    "payers.csv",
    "procedures.csv",
    "providers.csv",
    "supplies.csv"
]

print(f"--- Reading all column headers from {CSV_DIR} ---")
print("This will check all 18 files...\n")

missing_files = []

# Loop through each file
for file_name in csv_files_list:
    file_path = CSV_DIR / file_name

    # Check if file exists
    if not file_path.exists():
        print(f"!!! WARNING: File not found: {file_name} !!!\n")
        missing_files.append(file_name)
        continue

    # Read only the header row (nrows=0) to get columns
    try:
        df_header = pd.read_csv(file_path, nrows=0)

        print(f"--- Columns in {file_name} ---")
        print(df_header.columns.tolist())
        print("--------------------------------" + "-" * len(file_name) + "\n")

    except pd.errors.EmptyDataError:
        print(f"--- {file_name} is empty ---")
        print("[]")
        print("-----------------------" + "-" * len(file_name) + "\n")
    except Exception as e:
        print(f"!!! Error reading {file_name}: {e} !!!\n")

if missing_files:
    print(f"\nSummary: Could not find the following files: {missing_files}")
else:
    print("\nSummary: All 18 files were found and headers were read successfully.")

In [ ]:

import pandas as pd
from pathlib import Path
import os

# Path to your Synthea CSVs in Google Drive
CSV_DIR = Path("/content/drive/MyDrive/csv")
OUTPUT_DIR = Path("data/ehr")

def process_synthea_data_enhanced():
    """
    Reads multiple Synthea CSVs from Google Drive and creates one rich
    .txt file per patient in the Colab 'data/ehr/' directory.

    (Version 2 - Corrected DATE/START key error)
    """
    print(f"Reading CSVs from: {CSV_DIR}")

    if not CSV_DIR.exists():
        print(f"Error: Directory not found: {CSV_DIR}")
        return

    try:
        patients = pd.read_csv(CSV_DIR / "patients.csv")
        meds = pd.read_csv(CSV_DIR / "medications.csv")
        conditions = pd.read_csv(CSV_DIR / "conditions.csv")
        allergies = pd.read_csv(CSV_DIR / "allergies.csv")
        procedures = pd.read_csv(CSV_DIR / "procedures.csv")
        encounters = pd.read_csv(CSV_DIR / "encounters.csv")
        observations = pd.read_csv(CSV_DIR / "observations.csv")
    except FileNotFoundError as e:
        print(f"Error loading file: {e}")
        print(f"Please ensure all CSV files (patients, meds, conditions, etc.) are in your folder: {CSV_DIR}")
        return
    except Exception as e:
        print(f"An error occurred: {e}")
        return

    # Create output directory
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    print(f"Processing {len(patients)} patients...")

    # Process each patient
    for _, patient in patients.iterrows():
        patient_id = patient["Id"]

        # 1. Demographics
        patient_info = [
            f"Patient ID: {patient_id}",
            f"Name: {patient['FIRST']} {patient['LAST']}",
            f"Gender: {patient['GENDER']}",
            f"Birthdate: {patient['BIRTHDATE']}",
            f"Address: {patient.get('ADDRESS', 'N/A')}",
            f"Marital Status: {patient.get('MARITAL', 'N/A')}",
        ]

        # 2. Allergies
        patient_allergies = allergies[allergies["PATIENT"] == patient_id]
        allergy_list = [f"- {desc}" for desc in patient_allergies["DESCRIPTION"].unique()]

        # 3. Active Conditions
        patient_conditions = conditions[conditions["PATIENT"] == patient_id]
        condition_list = [f"- {desc}" for desc in patient_conditions["DESCRIPTION"].unique()]

        # 4. Current Medications
        patient_meds = meds[meds["PATIENT"] == patient_id]
        med_list = [f"- {desc}" for desc in patient_meds["DESCRIPTION"].unique()]

        # 5. Past Procedures
        patient_procs = procedures[procedures["PATIENT"] == patient_id]
        # FIX: Changed 'DATE' to 'START'
        proc_list = [f"- {row['DESCRIPTION']} (Date: {row['START']})" for _, row in patient_procs.iterrows()]

        # 6. Recent Encounters
        # FIX: Changed 'DATE' to 'START' for sorting
        patient_encs = encounters[encounters["PATIENT"] == patient_id].sort_values('START', ascending=False)
        # FIX: Changed 'row['DATE']' to 'row['START']'
        enc_list = [f"- {row['START']}: {row['DESCRIPTION']}" for _, row in patient_encs.head(5).iterrows()]

        # 7. Recent Observations (Vitals/Labs)
        # NO FIX NEEDED: 'DATE' column exists in observations.csv
        patient_obs = observations[observations["PATIENT"] == patient_id].sort_values('DATE', ascending=False)
        obs_list = [f"- {row['DATE']} {row['DESCRIPTION']}: {row['VALUE']} {row.get('UNITS', '')}" for _, row in patient_obs.head(10).iterrows()]

        # Assemble the text file content
        content = f"== PATIENT RECORD: {patient['FIRST']} {patient['LAST']} (ID: {patient_id}) ==\n\n"
        content += "== Demographics ==\n" + "\n".join(patient_info) + "\n\n"
        content += "== Allergies ==\n" + ("\n".join(allergy_list) if allergy_list else "None on record.") + "\n\n"
        content += "== Active Conditions / Problem List ==\n" + ("\n".join(condition_list) if condition_list else "None on record.") + "\n\n"
        content += "== Current Medications ==\n" + ("\n".join(med_list) if med_list else "None on record.") + "\n\n"
        content += "== Past Procedures ==\n" + ("\n".join(proc_list) if proc_list else "None on record.") + "\n\n"
        content += "== Recent Encounters (Last 5) ==\n" + ("\n".join(enc_list) if enc_list else "None on record.") + "\n\n"
        content += "== Recent Observations (Last 10) ==\n" + ("\n".join(obs_list) if obs_list else "None on record.") + "\n"

        # Write to file
        output_filename = OUTPUT_DIR / f"patient_{patient_id}.txt"
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(content)

    print(f"\nSuccessfully processed and saved {len(patients)} patient records to {OUTPUT_DIR}")
    print(f"Total files in {OUTPUT_DIR}: {len(list(OUTPUT_DIR.glob('*.txt')))}")

In [ ]:

process_synthea_data_enhanced()

# Check the output - let's find a random file and print it
print("\n--- Sample Enhanced EHR File ---")
!ls data/ehr | head -n 1 | xargs -I {} head -n 25 data/ehr/{}

In [ ]:

from datasets import load_dataset

try:
    # Ensure dataset is loaded
    mts_dataset = load_dataset("har1/MTS_Dialogue-Clinical_Note", split='train')

    # Print all column names
    print("--- Columns in MTS-Dialog Dataset ---")
    print(mts_dataset.column_names)
    print("-------------------------------------")

    # Print the first sample to see the structure
    print("\n--- First Sample Data ---")
    print(mts_dataset[0])
    print("---------------------------")

except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
from google.colab import files
import shutil

# Replace 'my_folder' with your folder name
shutil.make_archive('/content/data/ehr', 'zip', '/content/data/ehr')
files.download('/content/data/ehr.zip')


In [ ]:
!pip install evaluate sentence-transformers transformers torch

In [ ]:
!pip install rouge_score


In [ ]:


import pandas as pd
import numpy as np
import warnings
from evaluate import load
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoModel, AutoTokenizer
import torch

warnings.filterwarnings('ignore') # Suppress warnings during t-SNE

# --- Configuration (Adjust as Needed) ---
FILE_PATH = '/content/Mistral_7B_Checked.csv' # Your input data file
SEMANTIC_THRESHOLD = 0.8 # Threshold for "Semantic Accuracy"
NUM_CLUSTERS = 5 # Target number of error clusters for K-Means
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2' # Robust general-purpose embedding model

# Load Metrics and Model
rouge_metric = load('rouge')

# Initialize Embedding Model
try:
    tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL_NAME)
    model = AutoModel.from_pretrained(EMBEDDING_MODEL_NAME)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    print(f"Loaded embedding model onto: {device}")
except Exception as e:
    print(f"Error loading embedding model: {e}")
    def get_embeddings(texts): return np.zeros((len(texts), 384)) # Dummy fallback

# Function to generate embeddings
def get_embeddings(texts):
    """Generate sentence embeddings for a list of texts."""
    inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    # Mean pooling
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings
# %%

In [ ]:
# %%
# Load and select relevant columns
df = pd.read_csv(FILE_PATH)
df = df[['model', 'strategy', 'query_id', 'query', 'retrieved_context', 'answer', 'manual_review_factual_accuracy']]

# 1. Normalize Text
def clean_text(text):
    """Applies basic text normalization (lowercase, stripping) for metric comparison."""
    if isinstance(text, str):
        return text.lower().strip()
    return ""

df['answer_clean'] = df['answer'].apply(clean_text)

# 2. Add Gold Answer (REPLACE THIS WITH YOUR ACTUAL GOLD TRUTH DATA LOADING)
# Note: For the failed Q7 in the log, we use the validated ICD-10 J32.9 as a concrete example.
def get_simulated_gold(row):
    """Placeholder function for loading the actual gold truth."""
    if row['query_id'] == 'Q7':
        # Validated ICD-10 for Chronic sinusitis, unspecified
        return 'the icd-10 code for chronic sinusitis is j32.9 chronic sinusitis unspecified.'
    # WARNING: Using the model's answer as gold here (row['answer_clean']) inflates metrics.
    # Must be replaced with external ground truth.
    return row['answer_clean']

df['gold_answer'] = df.apply(get_simulated_gold, axis=1)

# 3. Define Failure Flags
df['is_manual_failure'] = (df['manual_review_factual_accuracy'] == 0.0)

print(f"Data Prepared. Total Samples: {len(df)}")
# %%

In [ ]:
# %%
# --- 1. Lexical Metrics (ROUGE) ---
predictions = df['answer_clean'].tolist()
# ROUGE expects a list of reference lists
references = [[g] for g in df['gold_answer'].tolist()]

# Compute ROUGE
rouge_results = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=True)

# FIX APPLIED HERE: Access 'rougeL' score directly
print(f"ROUGE-L F-Score (Lexical Overlap): {rouge_results['rougeL']:.4f}")

# Task-specific metric: Factual Exact Match
em_accuracy = (df['manual_review_factual_accuracy'] == 1.0).mean()
print(f"Factual Exact Match (based on manual review): {em_accuracy:.2%}")

# --- 2. Semantic Metrics (Cosine Similarity) ---
print("\nCalculating Semantic Embeddings...")
pred_embeddings = get_embeddings(df['answer_clean'].tolist())
gold_embeddings = get_embeddings(df['gold_answer'].tolist())

# Compute pairwise cosine similarity
semantic_scores = [
    cosine_similarity([pred_embeddings[i]], [gold_embeddings[i]])[0][0]
    for i in range(len(df))
]

df['semantic_similarity'] = semantic_scores
avg_semantic_similarity = df['semantic_similarity'].mean()
print(f"Average Semantic Similarity (Meaning Equivalence): {avg_semantic_similarity:.4f}")

# --- 3. Threshold Analysis (Semantic Accuracy) ---
df['semantic_accurate'] = df['semantic_similarity'] >= SEMANTIC_THRESHOLD
semantic_accuracy = df['semantic_accurate'].mean()
print(f"Semantic Accuracy (@{SEMANTIC_THRESHOLD} threshold): {semantic_accuracy:.2%}")
# %%

In [ ]:

# # RAG MLOps Productionization Pipeline (Post-Validation Focus)
#
# **Goal:** Transition an accurate RAG system (Mistral-7B) to a production-ready state by focusing on stability, robustness, and automated evaluation.
#
# **Assumption:** Factual Accuracy $\ge 80\%$.
#
# ---
#
# ## 1. Setup, Installations, and Configuration
#
# Run this cell once to install necessary libraries.

# %%
# !pip install pandas numpy scikit-learn evaluate rouge-score # Required MLOps/Metric libraries

import pandas as pd
import numpy as np
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from datetime import datetime

# --- CONFIGURATION (Based on rag_mlops_config.py) ---
TEST_DATA_PATH = '/content/Mistral_7B_Checked.csv'
NUM_REPEATS = 5           # N for Consistency Checks (Phase 1)
DECODING_TEMP = 0.25      # Recommended low temperature for stability
ACCURACY_ALERT_THRESHOLD = 0.80 # Threshold for Factual Accuracy alerting (Phase 4)

# --- UTILITY PLACEHOLDERS ---

def run_mistral_inference(query: str, context: str, seed: int, temperature: float) -> str:
    """
    SIMULATES calling the optimized Mistral-7B inference engine (e.g., vLLM/TGI).

    In a real system, this calls the serving endpoint with decoding controls.
    """
    np.random.seed(seed)

    # 1. Truncate context to keep input realistic and prompt-friendly
    context_snippet = context[:100].replace('\n', ' ')

    # 2. Simulate slightly different outputs (noise) based on seed

    if temperature < 0.3:
        # High stability expected for low temperature
        return f"A: The precise conclusion based on {context_snippet}... is: Final Answer (Stable, Seed={seed})."
    else:
        # Lower stability expected for higher temperature (e.g., in a robustness test)
        noise_level = np.random.uniform(-0.02, 0.02)
        return f"A: The potential finding is {context_snippet[:20]}... with some interpretation. Jitter: {noise_level:.2f} (Seed={seed})."


def get_semantic_similarity(answers: List[str]) -> Tuple[float, float]:
    """
    SIMULATES calculating mean pairwise cosine similarity and variance
    for a list of generated answers (Phase 1 Metric).
    """
    if len(answers) < 2:
        return 1.0, 0.0

    # Simulation based on assumed stable performance:
    base_sim = 0.95
    sim_noise = np.random.uniform(0.005, 0.015)
    mean_sim = base_sim - sim_noise
    variance = sim_noise * 10

    return mean_sim, variance

def get_llm_judge_score() -> float:
    """SIMULATES fetching the LLM-Judge Groundedness score (Phase 5 metric)."""
    # Simulate a high-performing RAG system score
    return np.random.uniform(0.92, 0.98)

def get_retrieval_recall() -> float:
    """SIMULATES fetching the Retrieval Recall@K (Phase 4 metric)."""
    # Simulate high performance
    return np.random.uniform(0.95, 0.99)


In [ ]:

# ## 2. Data Preparation and Ingestion (Simulated)

# %%
# Load and prepare data (Only selecting required columns for checks)
try:
    df = pd.read_csv(TEST_DATA_PATH).head(20) # Use a subset for faster simulation
    print(f"Data ingested successfully. Using {len(df)} samples for simulation.")
except FileNotFoundError:
    print(f"ERROR: File not found at {TEST_DATA_PATH}. Please ensure the file is uploaded.")
    df = pd.DataFrame({'query_id': [], 'query': [], 'retrieved_context': [], 'manual_review_factual_accuracy': []})




In [ ]:

# ## 3. Phase 1: Stability & Robustness Validation (Consistency Check)
#
# This checks the system's output stability when decoding hyperparameters are constrained for low variability.

# %%
def check_consistency(df_input: pd.DataFrame) -> pd.DataFrame:
    """
    Executes an N-repeat consistency check across a query set to measure
    the semantic stability (variance) of the RAG system's generation layer.
    """
    print(f"\n--- Running Phase 1: Consistency Check (N={NUM_REPEATS}, Temp={DECODING_TEMP}) ---")
    if df_input.empty:
        print("Input DataFrame is empty. Skipping consistency check.")
        return pd.DataFrame()

    results = []

    # Prepare list of inputs (query, context)
    test_queries = df_input[['query', 'retrieved_context']].values.tolist()

    for i, (query, context) in enumerate(test_queries):
        outputs = []
        for n in range(NUM_REPEATS):
            # 1. Run inference with fixed low temperature and unique seed
            answer = run_mistral_inference(
                query,
                context,
                seed=n + 1 + i,
                temperature=DECODING_TEMP
            )
            outputs.append(answer)

        # 2. Calculate mean pairwise similarity and variance (simulated)
        mean_similarity_score, variance_score = get_semantic_similarity(outputs)

        results.append({
            'query_id': df_input.iloc[i]['query_id'],
            'query': query,
            'mean_sim_score': mean_similarity_score,
            'variance_score': variance_score, # Key metric: Measures instability
            'sample_answers': outputs[0] # Store one sample output
        })

    df_results = pd.DataFrame(results)
    print(f"Consistency Check Complete. Processed {len(df_results)} queries.")
    return df_results

# Run the consistency check
consistency_results = check_consistency(df)
if not consistency_results.empty:
    print("\nConsistency Results Summary:")
    print(f"Average Output Variance: {consistency_results['variance_score'].mean():.4f}")
    # Display top 5 least consistent (highest variance) samples for manual review
    print("\nTop 3 Queries with Highest Variance:")
    print(consistency_results.sort_values('variance_score', ascending=False).head(3)[['query_id', 'variance_score']])



In [ ]:

# ## 4. Phase 4: Evaluation Automation (Nightly Regression Suite)
#
# This class simulates the automated nightly job, incorporating results from Phase 1, and logging core MLOps metrics.

# %%
class RAGRegressionSuite:
    """
    Simulates the automated nightly MLOps job, executing regression tests
    and logging metrics to an MLOps platform (e.g., MLflow).
    """
    def __init__(self, platform_name: str = 'MLflow'):
        self.platform = platform_name
        self.metrics: Dict[str, float] = {}

    def _execute_tests(self, df_test: pd.DataFrame, consistency_results: pd.DataFrame):
        """Core logic to run all production metrics."""

        # 1. Factual Accuracy (Baseline Factual Accuracy from the benchmark file)
        # Assuming the benchmark is accurate (as per user instruction)
        factual_accuracy = df_test['manual_review_factual_accuracy'].mean()

        # 2. Retrieval Recall@K (Simulated Phase 4 metric)
        retrieval_recall = get_retrieval_recall()

        # 3. Generation Fidelity (Simulated LLM-Judge Groundedness score - Phase 5)
        generation_fidelity = get_llm_judge_score()

        # 4. Consistency Variance (from Phase 1 check results)
        avg_consistency_variance = consistency_results['variance_score'].mean() if not consistency_results.empty else 0.0

        self.metrics = {
            'factual_accuracy': factual_accuracy,
            'retrieval_recall_at_k': retrieval_recall,
            'generation_fidelity': generation_fidelity,
            'consistency_variance': avg_consistency_variance,
            'processed_timestamp': datetime.utcnow().timestamp()
        }

    def run_nightly_job(self, data_path: str, consistency_results: pd.DataFrame):
        """Executes the full regression suite and orchestrates logging and alerting."""
        print(f"\n--- Running Phase 4: Nightly Regression Test ({self.platform}) ---")

        try:
            # 1. Data Ingestion
            df_test = pd.read_csv(data_path)
        except FileNotFoundError:
            print(f"ERROR: File not found at {data_path}. Cannot run nightly job.")
            return

        # 2. Execute tests
        self._execute_tests(df_test, consistency_results)

        # 3. Log results to MLOps platform (MLflow/W&B placeholders)
        print(f"Metrics Logged to {self.platform} Dashboard:")
        for k, v in self.metrics.items():
            if 'timestamp' not in k:
                print(f"  - {k}: {v:.4f}")

        # 4. Alerting Logic
        current_accuracy = self.metrics['factual_accuracy']
        if current_accuracy < ACCURACY_ALERT_THRESHOLD:
            print(f"\n🚨 ALERT! Factual Accuracy ({current_accuracy:.2%}) is below the threshold ({ACCURACY_ALERT_THRESHOLD:.2%}).")
            print("Action: Trigger immediate pipeline rollback and investigation.")
        else:
            print("\n SUCCESS. All production metrics passed. System is stable.")




In [ ]:
# ## 5. Main Pipeline Orchestration (Execute Job)

# %%
if __name__ == '__main__':
    print("===================================================================")
    print("  RAG MLOps Pipeline Orchestrator: Post-Validation Focus")
    print("===================================================================")

    # 1. Run Phase 1: Check stability and generate variance score
    # Note: df is defined in Block 2
    if not 'df' in locals() or df.empty:
        print("Skipping orchestration: Data not loaded in Block 2.")
    else:
        consistency_data = check_consistency(df)

        # 2. Run Phase 4: Execute the Nightly Regression Suite
        nightly_evaluator = RAGRegressionSuite(platform_name='MLflow')
        nightly_evaluator.run_nightly_job(
            data_path=TEST_DATA_PATH,
            consistency_results=consistency_data
        )
# %%



### Introduction
This analysis re-evaluates the notebook's execution and clarifies the factual accuracy metrics, distinguishing between the pre-existing `manual_review_factual_accuracy` from the benchmark file and the `programmatic accuracy` calculated using our custom comparison logic against the parsed EHR data.

### Analysis of Notebook Execution

**1. Environment Setup:**
*   **What was done**: Necessary Python libraries were installed, including data manipulation tools (`pandas`), NLP libraries (`sentence-transformers`, `transformers`), evaluation metrics (`evaluate`, `rouge-score`), and utilities for Colab (`ffmpeg`, `datasets`).
*   **Why it was done**: To prepare the Colab environment with all required dependencies for data processing, model evaluation, and MLOps simulation.
*   **Reason behind it**: Ensuring a functional environment before proceeding with any core tasks.

**2. Data Loading Attempts and Debugging:**
*   **What was done**: Initial attempts were made to load the `MTS_Dialogue-Clinical_Note` dataset. An initial `KeyError` was encountered due to an incorrect column name (`'note'`). Subsequent inspection revealed the correct column names, including `dialogue` and `section_text`.
*   **Why it was done**: This was part of exploring potential datasets, though ultimately this specific dataset was not used in the final accuracy comparison for the Synapse benchmark.
*   **Reason behind it**: Initial data exploration and debugging, which helped in understanding dataset structures.

**3. Accessing EHR Data:**
*   **What was done**: Google Drive was mounted, and the contents of the `/content/drive/MyDrive/csv` directory were listed.
*   **Why it was done**: To enable access to the raw Synthea CSV files containing the ground truth EHR data.
*   **Reason behind it**: Establishing connectivity to the primary data source for ground truth.

**4. Inspecting Raw EHR Data Schema:**
*   **What was done**: The column headers of all 18 Synthea CSV files were read and displayed.
*   **Why it was done**: To understand the schema and contents of the raw EHR data, which is essential for accurate parsing.
*   **Reason behind it**: Pre-processing step for data understanding and subsequent extraction.

**5. Processing Raw EHR Data into Patient-centric Text Files:**
*   **What was done**: The `process_synthea_data_enhanced` function was defined and executed. This function parsed various Synthea CSVs (patients, medications, conditions, etc.) and consolidated relevant information for each patient into individual `.txt` files in the `data/ehr/` directory.
*   **Why it was done**: To transform the disparate tabular EHR data into a unified, text-based format for each patient, mimicking clinical notes. This serves as the structured ground truth against which the model's answers would be programmatically validated.
*   **Reason behind it**: Creation of a standardized, patient-centric ground truth representation for evaluation.

**6. Archiving and Downloading Processed EHR Data:**
*   **What was done**: The `data/ehr` directory was compressed into a zip file and made available for download.
*   **Why it was done**: To provide an option for offline inspection or backup of the processed EHR data.
*   **Reason behind it**: Utility for data management.

**7. Initializing Embedding Model and ROUGE Metric:**
*   **What was done**: The `rouge` metric was loaded, and a `sentence-transformers/all-MiniLM-L6-v2` model was initialized for generating sentence embeddings. The model was configured to use a GPU if available.
*   **Why it was done**: These tools are used for quantitative evaluation of generated text. ROUGE measures lexical similarity, while sentence embeddings are used for semantic similarity, providing a measure of meaning overlap.
*   **Reason behind it**: Setting up the core NLP tools for performance measurement.

**8. Loading Benchmark Data and Preparing Gold Answers:**
*   **What was done**: The `Mistral_7B_Checked.csv` benchmark file was loaded. The `answer` column (model's output) was cleaned. A `gold_answer` column was created, primarily by duplicating the model's `answer` (with a note about inflated metrics) but specifically using a validated ICD-10 code for query Q7. This cell also highlighted the existing `manual_review_factual_accuracy` column.
*   **Why it was done**: This step ingests the results of the Mistral 7B model's performance on the benchmark queries, providing the answers to be evaluated. The `manual_review_factual_accuracy` within this dataset serves as a *reference point* for human-validated correctness.
*   **Reason behind it**: Ingesting the RAG model's outputs and associated manual quality scores for further analysis.

**9. Calculating Lexical and Semantic Metrics:**
*   **What was done**: ROUGE-L F-score (lexical overlap) was calculated. Sentence embeddings were generated for the cleaned model answers and the (partially simulated) `gold_answer`. Cosine similarity was then used to determine semantic similarity. This step also explicitly reported the **Factual Exact Match (based on manual review): 82.86%**, which directly references the `manual_review_factual_accuracy` column from the input CSV.
*   **Why it was done**: To provide an initial set of evaluation metrics (lexical and semantic) for the model's performance based on the provided benchmark data. It also presented the existing manual review score, which is a key piece of information for the user.
*   **Reason behind it**: Initial quantitative assessment of the model's performance based on direct comparison to the available benchmark values.

**10. RAG MLOps Productionization Pipeline Simulation:**
*   **What was done**: A simulated MLOps pipeline was implemented to showcase how a RAG system would be monitored in production. This included functions for simulating LLM inference stability (`check_consistency`) and a `RAGRegressionSuite` class for a nightly job. The nightly job reported various metrics, including `factual_accuracy` which explicitly referenced the `manual_review_factual_accuracy` column (resulting in 82.86%).
*   **Why it was done**: To demonstrate a framework for continuous monitoring and validation of a RAG system, going beyond one-off evaluations. It confirmed the model's high manual review accuracy as a baseline for the MLOps pipeline.
*   **Reason behind it**: Illustrating a robust system for ongoing RAG model quality assurance and performance tracking.

**11. Aligning Benchmark and EHR Data:**
*   **What was done**: Patient IDs were extracted from the `retrieved_context` column of the benchmark DataFrame. The benchmark DataFrame was then 'exploded' to link each query to potentially multiple patient IDs from its context. This expanded data was then merged with the parsed `df_ehr` DataFrame.
*   **Why it was done**: To create a unified dataset (`df_aligned`) where each model answer could be directly correlated with the specific EHR data points (from the `data/ehr` files) that were used to generate the answer. This is crucial for *programmatic validation* against the actual source data.
*   **Reason behind it**: Preparing the dataset for direct programmatic comparison of model answers against the ground truth EHR data.

**12. Defining Comparison Logic Functions:**
*   **What was done**: Three comparison functions (`compare_conditions`, `compare_medications`, `compare_allergies`) were defined, along with a `normalize_text` helper function. `normalize_text` converts text to lowercase, removes punctuation, and strips common medical qualifiers. The `compare_` functions use normalized text and keyword matching to assess if a model's answer is present in the EHR data for specific categories.
*   **Why it was done**: To create a robust, automated method for determining the factual correctness of the model's natural language answers against the structured EHR data, accounting for variations in phrasing and terminology.
*   **Reason behind it**: Implementing the core intelligence for automated factual accuracy evaluation.

**13. Executing Programmatic Accuracy Evaluation:**
*   **What was done**: A `determine_query_type` function was used to categorize queries (e.g., 'condition', 'medication', 'allergy'). The `df_aligned` DataFrame was iterated, and the appropriate `compare_` function was applied to determine `is_correct` for each entry. A `KeyError` with 'patient_id_ehr' was identified and corrected to 'patient_id'. Finally, programmatic accuracy was calculated per query type and overall.
*   **Why it was done**: This is the crucial step of *programmatically verifying* the model's answers against the EHR. This provides an automated, objective measure of how well the RAG system extracts and synthesizes information from the provided patient records. The error correction ensured that the correct patient identifier was used during the comparison.
*   **Reason behind it**: To obtain an automated, verifiable factual accuracy score directly from the EHR data, which can then be compared against any manual review scores.

### Conclusion and Summary

We have successfully established a comprehensive framework for evaluating the factual accuracy of a RAG model's outputs against detailed EHR data. This framework involves several key stages:

1.  **EHR Data Preparation**: Synthetic EHR data from Synthea was converted into structured, patient-centric text files, serving as our verifiable ground truth.
2.  **Benchmark Integration**: The Mistral 7B model's benchmark results, including its generated answers and a pre-existing **manual review factual accuracy of 82.86%**, were loaded and aligned with the EHR data.
3.  **Automated Comparison Logic**: Robust comparison functions were developed to programmatically assess the factual correctness of model answers against the EHR for various medical categories (conditions, medications, allergies), handling natural language variations.


**Key Findings & Discrepancies:**

*   The **manual review factual accuracy** for the Mistral 7B model is indeed **82.86%**, as provided in the benchmark file.

*   This discrepancy is critical: it suggests that either:
    *   Our programmatic comparison logic, while designed for robustness, might still be too strict or miss nuances that a human reviewer would understand (e.g., synonyms, indirect phrasing).
    *   The `retrieved_context` provided to the model in the benchmark might not perfectly align with *all* the information available in our parsed `data/ehr` files, leading to mismatches.
    *   The `gold_answer` used in the initial lexical/semantic metrics calculation (Cell 9), which largely mimicked the model's answer, naturally led to high initial scores (ROUGE-L 0.9180). This highlights the importance of truly independent ground truth for meaningful evaluation.
